# Verify that $\mathrm{tasmin} \leq \mathrm{tasmax}$

In dataset NEX-GDDP-CMIP6 some values of tasmin are greater than the respective tasmax values in the same day and this is not allowed logically.

The problem was first identified by Jacopo in data bias adjusted by me, but with further checks it seems to originate from the original data and my bias adjustment simply accentuates the absolute errors.

Since both NEX and I apply variants of quantile mapping method for bias adjustment, the hypothesis is that NEX applied bias adjustment to tasmin and tasmax variables separately and did not check for coherence of these two values after the adjustment. This can be considered a bug in the dataset.

Note that this problem should not be dependent on spatial and temporal dimensions, since they are just slices of the same model outputs. In other words, if the problem is present even for a single point in space and time of the dataset, that means that the entire dataset is subject to it and only due to randomness other points may not present the problem.

In [1]:
import pathlib

import numpy
import xarray

import settings
import thesis

In [2]:
dir_data = settings.dir_data / settings.location

def print_wrong(tasmin, tasmax):
    delta = tasmax - tasmin  # Should be always non negative.
    wrong = delta.where((delta < 0).compute(), other=numpy.nan)  # Set NaNs to use count.
    print(wrong.count(dim=("latitude", "longitude", "time")).values)  # Show spatial distribution for ease.

## Check coherence in reference data

Reference data (i.e. ERA5) do not present the error because tasmin and tasmax are evaluated directly from hourly tas data.

In [3]:
chunks = {"latitude": 1, "longitude": 1, "time": -1}
ds_reference = xarray.open_dataset(dir_data / "preprocessed" / "reference.nc", chunks=chunks)

print_wrong(ds_reference["tasmin"], ds_reference["tasmax"])

0


## Check coherence in all models after bias adjustment

Since the problem was initially identified in EC-Earth3 data after I applied quantile delta mapping algorithm, now I check which models present this problem among the ones I use in the thesis.

In [4]:
m_dim = "model"
ds_historical_ba = xarray.open_mfdataset(
    [dir_data / "adjusted" / f"historical_{m}.nc" for m in thesis.models],
    chunks=chunks,
    concat_dim=m_dim,
    combine="nested",
    combine_attrs="drop_conflicts"
)
ds_historical_ba = ds_historical_ba.chunk({m_dim: -1})
ds_historical_ba = ds_historical_ba.assign_coords({m_dim: list(thesis.models)})

s_dim = "scenario"
ds_scenario_ba = xarray.open_mfdataset(
    [[dir_data / "adjusted" / f"{s}_{m}.nc" for m in thesis.models] for s in thesis.scenario],
    chunks=chunks,
    concat_dim=[s_dim, m_dim],
    combine="nested",
    combine_attrs="drop_conflicts"
)
ds_scenario_ba = ds_scenario_ba.chunk({s_dim: 1, m_dim: -1})
ds_scenario_ba = ds_scenario_ba.assign_coords({s_dim: list(thesis.scenario), m_dim: list(thesis.models)})

In [5]:
print("historical")
for m in ds_historical_ba["model"].values:
    print(m)
    ds = ds_historical_ba.sel(model=m)
    print_wrong(ds["tasmin"], ds["tasmax"])
print()

for s in ds_scenario_ba["scenario"].values:
    print(s)
    for m in ds_scenario_ba["model"].values:
        print(m)
        ds = ds_scenario_ba.sel(scenario=s, model=m)
        print_wrong(ds["tasmin"], ds["tasmax"])
    print()

historical
ACCESS-CM2
4
ACCESS-ESM1-5
13
BCC-CSM2-MR
1
CanESM5
0
CMCC-ESM2
0
CNRM-CM6-1
6
CNRM-ESM2-1
1
EC-Earth3
11
EC-Earth3-Veg-LR
3
FGOALS-g3
0
GFDL-ESM4
1
INM-CM4-8
2
INM-CM5-0
3
IPSL-CM6A-LR
0
MIROC-ES2L
0
MIROC6
0
MPI-ESM1-2-HR
17
MPI-ESM1-2-LR
0
MRI-ESM2-0
19
NorESM2-LM
0
NorESM2-MM
3
TaiESM1
0

ssp126
ACCESS-CM2
0
ACCESS-ESM1-5
3
BCC-CSM2-MR
6
CanESM5
0
CMCC-ESM2
0
CNRM-CM6-1
0
CNRM-ESM2-1
3
EC-Earth3
27
EC-Earth3-Veg-LR
7
FGOALS-g3
0
GFDL-ESM4
0
INM-CM4-8
2
INM-CM5-0
5
IPSL-CM6A-LR
0
MIROC-ES2L
0
MIROC6
0
MPI-ESM1-2-HR
19
MPI-ESM1-2-LR
0
MRI-ESM2-0
8
NorESM2-LM
0
NorESM2-MM
0
TaiESM1
0

ssp245
ACCESS-CM2
0
ACCESS-ESM1-5
2
BCC-CSM2-MR
2
CanESM5
0
CMCC-ESM2
1
CNRM-CM6-1
3
CNRM-ESM2-1
0
EC-Earth3
33
EC-Earth3-Veg-LR
15
FGOALS-g3
1
GFDL-ESM4
0
INM-CM4-8
4
INM-CM5-0
3
IPSL-CM6A-LR
0
MIROC-ES2L
0
MIROC6
0
MPI-ESM1-2-HR
31
MPI-ESM1-2-LR
0
MRI-ESM2-0
8
NorESM2-LM
0
NorESM2-MM
0
TaiESM1
0

ssp370
ACCESS-CM2
0
ACCESS-ESM1-5
6
BCC-CSM2-MR
7
CanESM5
0
CMCC-ESM2
1
CNRM-CM6-1
1
CNRM-ESM2-1

## Check coherence for all models before bias adjustment

Is the cause of the problem the bias adjustment I apply to NEX-GDDP-CMIP6 data?
To answer this question, first I need to check that the original data did not present the problem.

Instead, as can be seen below, also NEX-GDDP-CMIP6 original data is affected by this problem (the preprocessing I applied introduces no changes in the values and their coordinates, for a proof check the content of file `debug_tas_extremes.txt` which is the same test as below but performed on data before they was preprocessed).

In [6]:
ds_historical = xarray.open_mfdataset(
    [dir_data / "preprocessed" / f"historical_{m}.nc" for m in thesis.models],
    chunks=chunks,
    concat_dim=m_dim,
    combine="nested",
    combine_attrs="drop_conflicts"
)
ds_historical = ds_historical.chunk({m_dim: -1})
ds_historical = ds_historical.assign_coords({m_dim: list(thesis.models)})

ds_scenario = xarray.open_mfdataset(
    [[dir_data / "preprocessed" / f"{s}_{m}.nc" for m in thesis.models] for s in thesis.scenario],
    chunks=chunks,
    concat_dim=[s_dim, m_dim],
    combine="nested",
    combine_attrs="drop_conflicts"
)
ds_scenario = ds_scenario.chunk({s_dim: 1, m_dim: -1})
ds_scenario = ds_scenario.assign_coords({s_dim: list(thesis.scenario), m_dim: list(thesis.models)})

In [7]:
print("historical")
for m in ds_historical["model"].values:
    print(m)
    ds = ds_historical.sel(model=m)
    print_wrong(ds["tasmin"], ds["tasmax"])
print()

for s in ds_scenario["scenario"].values:
    print(s)
    for m in ds_scenario["model"].values:
        print(m)
        ds = ds_scenario.sel(scenario=s, model=m)
        print_wrong(ds["tasmin"], ds["tasmax"])
    print()

historical
ACCESS-CM2
0
ACCESS-ESM1-5
0
BCC-CSM2-MR
0
CanESM5
0
CMCC-ESM2
0
CNRM-CM6-1
0
CNRM-ESM2-1
0
EC-Earth3
0
EC-Earth3-Veg-LR
0
FGOALS-g3
0
GFDL-ESM4
0
INM-CM4-8
1
INM-CM5-0
1
IPSL-CM6A-LR
0
MIROC-ES2L
0
MIROC6
0
MPI-ESM1-2-HR
0
MPI-ESM1-2-LR
0
MRI-ESM2-0
0
NorESM2-LM
1
NorESM2-MM
0
TaiESM1
0

ssp126
ACCESS-CM2
0
ACCESS-ESM1-5
0
BCC-CSM2-MR
0
CanESM5
0
CMCC-ESM2
0
CNRM-CM6-1
0
CNRM-ESM2-1
0
EC-Earth3
2
EC-Earth3-Veg-LR
8
FGOALS-g3
0
GFDL-ESM4
0
INM-CM4-8
0
INM-CM5-0
3
IPSL-CM6A-LR
0
MIROC-ES2L
0
MIROC6
0
MPI-ESM1-2-HR
5
MPI-ESM1-2-LR
0
MRI-ESM2-0
0
NorESM2-LM
0
NorESM2-MM
7
TaiESM1
0

ssp245
ACCESS-CM2
0
ACCESS-ESM1-5
0
BCC-CSM2-MR
0
CanESM5
0
CMCC-ESM2
1
CNRM-CM6-1
0
CNRM-ESM2-1
0
EC-Earth3
11
EC-Earth3-Veg-LR
13
FGOALS-g3
1
GFDL-ESM4
0
INM-CM4-8
1
INM-CM5-0
3
IPSL-CM6A-LR
0
MIROC-ES2L
0
MIROC6
0
MPI-ESM1-2-HR
1
MPI-ESM1-2-LR
0
MRI-ESM2-0
0
NorESM2-LM
0
NorESM2-MM
1
TaiESM1
0

ssp370
ACCESS-CM2
0
ACCESS-ESM1-5
0
BCC-CSM2-MR
0
CanESM5
0
CMCC-ESM2
3
CNRM-CM6-1
0
CNRM-ESM2-1
0
EC-E

The temporary fix I implement is to reassign the values of tasmin, tasmax and tas based on their relative order,  in each day and spatial coordinate. For instance, if for a given day and grid point the values are

| tasmin | tas   | tasmax |
|--------|-------|--------|
| 26 °C  | 34 °C | 16 °C  |

hence $\mathrm{tasmax} \leq \mathrm{tasmin} \leq \mathrm{tas}$, then I reorder these values and reassign the variables names to keep coherence with their meaning (i.e. $\mathrm{tasmin} \leq \mathrm{tas} \leq \mathrm{tasmax}$), resulting in

| tasmin | tas   | tasmax |
|--------|-------|--------|
| 16 °C  | 26 °C | 34 °C  |

From a physical and statistical point of view this procedure is not correct since it is changing the probability distribution of these variables, but it is straightforward to apply and brings coherent results.

This fix should be applied after the application of bias adjustment, since each iteration of this method introduces possible incoherence between their values.